In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter
from qiskit.quantum_info import SparsePauliOp

from qiskit_aer import Aer
from qiskit.visualization import array_to_latex
import qiskit.quantum_info as qi
import numpy as np
import scipy as sp

In [2]:
X = SparsePauliOp("X")
Z = SparsePauliOp("Z")
I = SparsePauliOp("I")

final_time = 20.0
num_timesteps = 10

dt = final_time / num_timesteps
H = (Z ^ Z) - 0.1 * (X ^ I)

#### 1)

In [3]:
trotter_step_first_order = PauliEvolutionGate(H, dt, synthesis=LieTrotter())

# We create an empty circuit
circuit = QuantumCircuit(H.num_qubits)
initial_state = qi.Statevector.from_circuit(circuit)
for i in range(num_timesteps):
    circuit.append(trotter_step_first_order, range(H.num_qubits))
    circuit.barrier()
circuit = circuit.decompose(reps=2)

# Let us print some stats
print(
    f"""
Trotter step with Lie-Trotter
-----------------------------

                  Depth: {circuit.depth()}
             Gate count: {len(circuit)}
    Nonlocal gate count: {circuit.num_nonlocal_gates()}
         Gate breakdown: {", ".join([f"{k.upper()}: {v}" for k, v in circuit.count_ops().items()])}
"""
)

# And finally draw the circuit
#circuit.draw("mpl")


Trotter step with Lie-Trotter
-----------------------------

                  Depth: 40
             Gate count: 50
    Nonlocal gate count: 20
         Gate breakdown: CX: 20, RZ: 10, R: 10, BARRIER: 10



In [4]:
backend = Aer.get_backend('statevector_simulator')
final_state = backend.run(circuit).result().get_statevector()
array_to_latex(final_state, precision=6)

<IPython.core.display.Latex object>

In [5]:
op = H.to_matrix()
exact_times = np.linspace(0, final_time, 101)
exact_evolution = [initial_state.evolve(sp.linalg.expm(-1j * time * op)) for time in exact_times]
array_to_latex(exact_evolution[-1], precision=6)

<IPython.core.display.Latex object>

In [6]:
print("State fidelity:", qi.state_fidelity(final_state, exact_evolution[-1]))

State fidelity: 0.9351119121777253


#### 2)

In [7]:
from qiskit.synthesis import SuzukiTrotter

second_order_formula = SuzukiTrotter()  # if not passed, order defaults to 2
trotter_step_second_order = PauliEvolutionGate(H, dt, synthesis=second_order_formula)
circuit = QuantumCircuit(H.num_qubits)
for i in range(num_timesteps):
    circuit.append(trotter_step_second_order, range(H.num_qubits))
    circuit.barrier()
circuit = circuit.decompose(reps=2)

# Let us print some stats
print(
    f"""
Trotter step with Suzuki Trotter (2nd order)
--------------------------------------------

                  Depth: {circuit.depth()}
             Gate count: {len(circuit)}
    Nonlocal gate count: {circuit.num_nonlocal_gates()}
         Gate breakdown: {", ".join([f"{k.upper()}: {v}" for k, v in circuit.count_ops().items()])}

"""
)

# And finally
#circuit.draw("mpl")


Trotter step with Suzuki Trotter (2nd order)
--------------------------------------------

                  Depth: 70
             Gate count: 80
    Nonlocal gate count: 40
         Gate breakdown: CX: 40, RZ: 20, R: 10, BARRIER: 10




In [8]:
backend = Aer.get_backend('statevector_simulator')
final_state = backend.run(circuit).result().get_statevector()
array_to_latex(final_state, precision=6)

<IPython.core.display.Latex object>

In [9]:
array_to_latex(exact_evolution[-1], precision=6)

<IPython.core.display.Latex object>

In [10]:
print("State fidelity:", qi.state_fidelity(final_state, exact_evolution[-1]))

State fidelity: 0.9900520165706168


#### 3)

In [11]:
fourth_order_formula = SuzukiTrotter(order=4)
trotter_step_fourth_order = PauliEvolutionGate(H, dt, synthesis=fourth_order_formula)
circuit = QuantumCircuit(H.num_qubits)
for i in range(num_timesteps):
    circuit.append(trotter_step_fourth_order, range(H.num_qubits))
    circuit.barrier()
circuit = circuit.decompose(reps=2)

# Let us print some stats
print(
    f"""
Trotter step with Suzuki Trotter (4th order)
--------------------------------------------

                  Depth: {circuit.depth()}
             Gate count: {len(circuit)}
    Nonlocal gate count: {circuit.num_nonlocal_gates()}
         Gate breakdown: {", ".join([f"{k.upper()}: {v}" for k, v in circuit.count_ops().items()])}

"""
)


Trotter step with Suzuki Trotter (4th order)
--------------------------------------------

                  Depth: 350
             Gate count: 360
    Nonlocal gate count: 200
         Gate breakdown: CX: 200, RZ: 100, R: 50, BARRIER: 10




In [12]:
backend = Aer.get_backend('statevector_simulator')
final_state = backend.run(circuit).result().get_statevector()
array_to_latex(final_state, precision=6)

<IPython.core.display.Latex object>

In [13]:
array_to_latex(exact_evolution[-1], precision=6)

<IPython.core.display.Latex object>

In [14]:
print("State fidelity:", qi.state_fidelity(final_state, exact_evolution[-1]))

State fidelity: 0.9998543205286909


In [16]:
! python -c 'import qiskit; print(qiskit.__version__);'

2.6.2


In [18]:
import tutorial_magics

%qiskit_version_table
#%qiskit_copyright